# Планы и производительность — 30 заданий

Практика на eBay; решений нет.

## Результаты обучения

После **Производительность** вы должны объяснить transformation как plan, предсказать action/jobs/shuffle, связать schema/grain с результатом и доказать физическую эффективность через explain/UI/metrics.

## Ментальная модель исполнения

Catalyst строит optimized logical и physical plan; AQE использует runtime statistics. Exchange, Sort, spill, skew и Python UDF объясняют основную стоимость.

```text
transformations → unresolved logical plan
       ↓ analysis (catalog/types)
   optimized logical plan (Catalyst)
       ↓ physical planning / AQE
job → stage → shuffle → stage
       tasks             tasks
       └──── executors ─────┘
```
Action создаёт job. Один notebook/application может породить много jobs, а один job —
несколько stages. `repartition`, join и groupBy часто добавляют Exchange.

## Данные eBay

Grain eBay — `itemid` в `snapshot_dt`; 2 501 511 строк, 24 колонки, Parquet/Snappy.
Partition column — дата снимка. Цена, продавец, категории и доставка денормализованы.
Перед `latest item` или dedup проверяйте уникальность пары и задавайте tie-breaker.

Полная схема и проверки качества находятся в `data-catalog`. Raw read-only, результаты — в личном `spark_training`.

## Алгоритм решения

1. Зафиксируйте входной и целевой grain. 2. Выберите только нужные columns/rows. 3. Соберите transformation без action. 4. Проверьте schema и explain. 5. Предскажите partitions/shuffle. 6. Выполните минимальный action/write. 7. Повторно прочитайте и сверяйте keys/metrics. 8. Сохраните evidence.

Сначала baseline и correctness, затем один change, повторный benchmark и plan evidence.

## Типичные ошибки

- Вызывать count/show после каждого шага и создавать лишние jobs.
- Использовать Python UDF при наличии встроенной функции.
- Делать repartition без понимания Exchange и целевого файла.
- Broadcast большой стороны или collect на driver.
- Кэшировать одноразовый DataFrame без materialization/unpersist.
- Измерять скорость при разных результатах или непрогретом JVM.

## Самопроверка

1. Какой action создаёт job? 2. Где появится shuffle? 3. Сколько input/output partitions? 4. Видит ли Catalyst выражение? 5. Каков grain после JOIN/window? 6. Как проверить idempotent rerun?

## Подробная теория

### 1. Планы

Parsed, analyzed, optimized logical и physical plan отвечают на разные вопросы.

### 2. Catalyst

Pushdown, pruning и constant folding работают для понятных engine выражений.

### 3. AQE

Runtime statistics позволяют сменить join, объединить shuffle partitions и обработать skew.

### 4. Memory

Cache материализуется action; spill означает перенос части работы на диск.

### 5. Benchmark

Сравнивайте одинаковый результат, прогревайте JVM и повторяйте измерение.

## Сдача

Каждое задание записывает непустой Parquet в личный HDFS и evidence с transformation, observation и explanation. Checker использует активную SparkSession.

In [ ]:
import os,sys
sys.path.insert(0,'/opt/lab/spark-training')
from check_task import check_task,save_evidence
from pyspark.sql import SparkSession,functions as F,types as T,Window
spark=SparkSession.builder.appName('spark-training').enableHiveSupport().getOrCreate()
USER=os.environ.get('HDFS_USER',os.environ.get('HADOOP_USER_NAME','student'))
ROOT=f'hdfs://namenode:8020/user/{USER}/spark_training'
SOURCE='hdfs://namenode:8020/data/raw/ebay'
ebay=spark.read.parquet(SOURCE)
print('Spark',spark.version,'rows',ebay.count(),'columns',len(ebay.columns))

### Задание 1. explain simple

Создайте результат по теме **explain simple** и запишите `mode("overwrite").parquet(f"{ROOT}/performance/task_01")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'performance',1)

### Задание 2. explain formatted

Создайте результат по теме **explain formatted** и запишите `mode("overwrite").parquet(f"{ROOT}/performance/task_02")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'performance',2)

### Задание 3. logical plan

Создайте результат по теме **logical plan** и запишите `mode("overwrite").parquet(f"{ROOT}/performance/task_03")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'performance',3)

### Задание 4. optimized plan

Создайте результат по теме **optimized plan** и запишите `mode("overwrite").parquet(f"{ROOT}/performance/task_04")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'performance',4)

### Задание 5. physical plan

Создайте результат по теме **physical plan** и запишите `mode("overwrite").parquet(f"{ROOT}/performance/task_05")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'performance',5)

### Задание 6. Catalyst rules

Создайте результат по теме **Catalyst rules** и запишите `mode("overwrite").parquet(f"{ROOT}/performance/task_06")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'performance',6)

### Задание 7. statistics

Создайте результат по теме **statistics** и запишите `mode("overwrite").parquet(f"{ROOT}/performance/task_07")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'performance',7)

### Задание 8. AQE

Создайте результат по теме **AQE** и запишите `mode("overwrite").parquet(f"{ROOT}/performance/task_08")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'performance',8)

### Задание 9. shuffle partitions

Создайте результат по теме **shuffle partitions** и запишите `mode("overwrite").parquet(f"{ROOT}/performance/task_09")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'performance',9)

### Задание 10. broadcast threshold

Создайте результат по теме **broadcast threshold** и запишите `mode("overwrite").parquet(f"{ROOT}/performance/task_10")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'performance',10)

### Задание 11. broadcast hint

Создайте результат по теме **broadcast hint** и запишите `mode("overwrite").parquet(f"{ROOT}/performance/task_11")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'performance',11)

### Задание 12. join strategies

Создайте результат по теме **join strategies** и запишите `mode("overwrite").parquet(f"{ROOT}/performance/task_12")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'performance',12)

### Задание 13. sort merge join

Создайте результат по теме **sort merge join** и запишите `mode("overwrite").parquet(f"{ROOT}/performance/task_13")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'performance',13)

### Задание 14. broadcast hash join

Создайте результат по теме **broadcast hash join** и запишите `mode("overwrite").parquet(f"{ROOT}/performance/task_14")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'performance',14)

### Задание 15. shuffle hash join

Создайте результат по теме **shuffle hash join** и запишите `mode("overwrite").parquet(f"{ROOT}/performance/task_15")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'performance',15)

### Задание 16. nested loop

Создайте результат по теме **nested loop** и запишите `mode("overwrite").parquet(f"{ROOT}/performance/task_16")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'performance',16)

### Задание 17. exchange node

Создайте результат по теме **exchange node** и запишите `mode("overwrite").parquet(f"{ROOT}/performance/task_17")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'performance',17)

### Задание 18. sort node

Создайте результат по теме **sort node** и запишите `mode("overwrite").parquet(f"{ROOT}/performance/task_18")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'performance',18)

### Задание 19. filter pushdown

Создайте результат по теме **filter pushdown** и запишите `mode("overwrite").parquet(f"{ROOT}/performance/task_19")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'performance',19)

### Задание 20. column pruning plan

Создайте результат по теме **column pruning plan** и запишите `mode("overwrite").parquet(f"{ROOT}/performance/task_20")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'performance',20)

### Задание 21. cache plan

Создайте результат по теме **cache plan** и запишите `mode("overwrite").parquet(f"{ROOT}/performance/task_21")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'performance',21)

### Задание 22. cache materialization

Создайте результат по теме **cache materialization** и запишите `mode("overwrite").parquet(f"{ROOT}/performance/task_22")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'performance',22)

### Задание 23. storage level

Создайте результат по теме **storage level** и запишите `mode("overwrite").parquet(f"{ROOT}/performance/task_23")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'performance',23)

### Задание 24. skew detection

Создайте результат по теме **skew detection** и запишите `mode("overwrite").parquet(f"{ROOT}/performance/task_24")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'performance',24)

### Задание 25. salting

Создайте результат по теме **salting** и запишите `mode("overwrite").parquet(f"{ROOT}/performance/task_25")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'performance',25)

### Задание 26. repartition by range

Создайте результат по теме **repartition by range** и запишите `mode("overwrite").parquet(f"{ROOT}/performance/task_26")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'performance',26)

### Задание 27. avoid UDF

Создайте результат по теме **avoid UDF** и запишите `mode("overwrite").parquet(f"{ROOT}/performance/task_27")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'performance',27)

### Задание 28. pandas UDF concept

Создайте результат по теме **pandas UDF concept** и запишите `mode("overwrite").parquet(f"{ROOT}/performance/task_28")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'performance',28)

### Задание 29. benchmark

Создайте результат по теме **benchmark** и запишите `mode("overwrite").parquet(f"{ROOT}/performance/task_29")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'performance',29)

### Задание 30. performance audit

Создайте результат по теме **performance audit** и запишите `mode("overwrite").parquet(f"{ROOT}/performance/task_30")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'performance',30)